# True-colour approach 4 — mapping newly exposed riverbed at Novi Sad

The idea: freshly exposed sand is *bright* in true colour, open water is *dark*, so a pixel that goes
from dark (water) to bright (sand) between spring and July is newly exposed riverbed. But brightness
alone is fooled by farmland (green April fields → bright harvested July fields read as "exposed" too).

The fix is to **bound the search to the river channel**: we use NDWI (a one-line water index from the
green and NIR bands) to outline the *spring* water extent, then map, within that channel, the pixels
that have since dried out — displayed on the July true-colour image. NDWI only defines "where the river
was"; the drying itself is what we map.

> Needs `GEE_SERVICE_ACCOUNT` / `GEE_SERVICE_KEY` and `pyramids-gis[viz]`.

## Setup

`pyramids` reads and plots the GeoTIFFs (`Dataset` / `DatasetCollection`); `earthlens` provides the
`EarthLens` entry point. Earth Engine credentials come from a repo-root `.env`.

In [ ]:
import os
import shutil
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from loguru import logger
from pyramids.dataset import Dataset, GeoReference

from earthlens.core import EarthLens

warnings.filterwarnings("ignore")
logger.remove()
plt.rcParams["figure.dpi"] = 80

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
SERVICE_ACCOUNT = os.environ["GEE_SERVICE_ACCOUNT"]
SERVICE_KEY = os.environ["GEE_SERVICE_KEY"]

## The reach and the two composites

The Danube at Novi Sad (braided, sandbar-rich). We take an early-season ("wet") and a July ("dry")
`median` composite with the visible bands plus NIR (`B8`) so we can compute NDWI, at 15 m.

In [ ]:
AOI = [19.78, 45.20, 19.95, 45.30]  # Danube at Novi Sad
S2 = "COPERNICUS/S2_SR_HARMONIZED"
SCALE = 15  # 15 m keeps the export under GEE's 50 MB cap
COMPOSITES = {"wet": ("2026-04-01", "2026-05-15"), "dry": ("2026-07-01", "2026-07-24")}
OUT = Path("out") / "tc4_novisad_exposed"
OUT.mkdir(parents=True, exist_ok=True)

## Fetch the wet and dry composites (RGB + NIR, cached)

In [ ]:
paths = {}
for tag, (start, end) in COMPOSITES.items():
    p = OUT / f"novisad_{tag}.tif"
    if not p.exists():
        job = EarthLens(
            data_source="gee",
            cadence="raw",
            path=tempfile.mkdtemp(),
            dataset=S2,
            variables=["B4", "B3", "B2", "B8"],
            aoi=AOI,
            start=start,
            end=end,
            scale=SCALE,
            reducer="median",
        )
        job.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
        out = job.download(progress_bar=False)
        if not out:
            continue
        shutil.copy(str(out[0]), p)
    paths[tag] = p
{k: v.name for k, v in paths.items()}

## Derive the exposed-riverbed mask (NDWI-bounded)

NDWI = (green - NIR) / (green + NIR); it is positive over open water. The **spring channel** is where
NDWI was positive in the wet composite; **exposed riverbed** is that channel where NDWI has since gone
negative (no longer water). Bounding by the spring channel is what keeps farmland out of the result.

In [ ]:
# Keep the July scene's Dataset: the true-colour panels below are placed
# on its grid rather than on pixel indices.
dry_source = Dataset.read_file(paths["dry"])
wet = np.asarray(Dataset.read_file(paths["wet"]).read_array(), dtype="float32")
dry = np.asarray(dry_source.read_array(), dtype="float32")


def ndwi(a):  # bands: 0=B4 red, 1=B3 green, 2=B2 blue, 3=B8 NIR
    return (a[1] - a[3]) / (a[1] + a[3] + 1e-6)


channel = ndwi(wet) > 0  # water in spring
still_water = ndwi(dry) > 0  # water in July
exposed = channel & ~still_water  # spring channel that has since dried
print(
    "spring channel px:",
    int(channel.sum()),
    "| exposed px:",
    int(exposed.sum()),
    "| exposed fraction of channel:",
    round(float(exposed.sum() / max(channel.sum(), 1)), 3),
)

## Map it: July true colour with the exposed bed highlighted

The July composite in true colour, with the newly exposed riverbed overlaid in orange — confined to the
channel, so farmland no longer contaminates it.

In [ ]:
rgb = np.clip(
    np.stack([dry[0], dry[1], dry[2]], axis=-1) / np.nanpercentile(dry[:3], 98), 0, 1
)
overlay = np.zeros((*exposed.shape, 4), dtype="float32")
overlay[exposed] = (1.0, 0.5, 0.0, 0.9)  # orange where exposed

# The composite is already stretched to 0..1, so it is wrapped as three bands
# and drawn unchanged: same pixels, on the scene's own coordinates.
truecolour = Dataset.from_array(
    np.moveaxis(rgb, -1, 0),
    no_data_value=np.nan,
    geo_ref=GeoReference(geo=dry_source.geotransform, epsg=dry_source.epsg),
)

# matplotlib lays the pair out; pyramids draws both georeferenced panels.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
truecolour.plot(
    fig=fig,
    ax=axes[0],
    rgb_options={"rgb": [0, 1, 2]},
    title="Danube at Novi Sad - July true colour",
)
panel = truecolour.plot(
    fig=fig,
    ax=axes[1],
    rgb_options={"rgb": [0, 1, 2]},
    title="newly exposed riverbed (orange)",
)
# The mask is an annotation over the rendered panel, so it is drawn onto the
# glyph's axes using that panel's own extent -- it stays aligned with the raster.
panel.ax.imshow(overlay, extent=panel.im.get_extent(), origin="upper")
plt.tight_layout()

## Notes

- The honest lesson: **pure-RGB brightness can't separate riverbed from harvested farmland** — both go
  dark-to-bright. Bounding the search to the spring water channel (one NDWI line) fixes that.
- The sediment story is still shown in true colour; NDWI only supplies the channel outline.